1. Read files from cloud storage using datastream reader API
2. Transform dataframe to add column
3. write transformed data steam into delta lake table

###Read files

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType
customer_schema = StructType(fields=[
  StructField("customer_id", IntegerType()),
  StructField("customer_name", StringType()),
  StructField("date_of_birth", DateType()),
  StructField("telephone", StringType()),
  StructField("email", StringType()),
  StructField("member_since", DateType()),
  StructField("created_timestamp", StringType())]
)
  

In [0]:
df = spark.readStream.format("json").schema(customer_schema).load("/Volumes/gizmobox/landing/operational_data/customer_stream/")

###transform dataframe to add columns

In [0]:
from pyspark.sql import functions as F
df1 = df.withColumn("file_path",F.col("_metadata.file_path")).withColumn("ingestion_date",F.current_timestamp())

###Write the transformed data to delta table

In [0]:
streaming_query = df1.writeStream.format("delta").option("checkpointlocation","/Volumes/gizmobox/landing/operational_data/customer_stream/_checkpoint_stream").toTable("gizmobox.bronze.customers_stream")

In [0]:
streaming_query.stop()

In [0]:
%sql
select * from gizmobox.bronze.customers_stream